In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, r2_score
from math import sqrt
import ast

In [ ]:
def process_df(df, feature_code):
    # feature_code: [old/new/no]-pol_[old/new/no]-pi_[y/n]-ecfp_[all/no/x/y/z]-torsion_[all/no/x/y/z]-sd
    if feature_code.endswith('_pt'):
        feature_code = feature_code.split('_pt')[0]
        first_cols = ['Monomer 1', 'Monomer 2']
    else:
        first_cols = ['Monomer 1', 'Monomer 2', 'Oligomer Type']
        
    pol, pi, ecfp = feature_code.split('_')[:-2]
    torsion_selection = feature_code.split('_')[-2]
    sd_selection = feature_code.split('_')[-1]
    
    geo_options = ['x', 'y', 'z']
    torsion_options = ['Total Torsion Angle', 'Mean Torsion Angle', 'Inner Mean Torsion Angle']
    sd_options = ['SD Torsions', 'SD Difference']
    
    geo_select_dict = {
        'all-torsion': [f"{t_opt}_{g_opt}" for t_opt in torsion_options for g_opt in geo_options],
        'all-sd': [f"{s_opt}_{g_opt}" for s_opt in sd_options for g_opt in geo_options],
        'no-torsion': [],
        'no-sd': [],
        'x-torsion': [f"{t_opt}_x" for t_opt in torsion_options],
        'y-torsion': [f"{t_opt}_y" for t_opt in torsion_options],
        'z-torsion': [f"{t_opt}_z" for t_opt in torsion_options],
        'x-sd': [f"{s_opt}_x" for s_opt in sd_options],
        'y-sd': [f"{s_opt}_y" for s_opt in sd_options],
        'z-sd': [f"{s_opt}_z" for s_opt in sd_options],
    }
    
    others_select_dict = {
        'old-pol': ['Molecule Polarizability', 'Average Monomer Polarizability', 'Dipole Moment'],
        'new-pol': ['Rotated Polarizability', 'Average Monomer Polarizability', 'Dipole Moment', 'α_xx', 'α_xy', 'α_yy'],
        'old-pi': ['PiSystemSize'],
        'new-pi': ['PiSystemSize', 'PiSystemArea', 'PiSystemLength'],
        'y-ecfp': [f"ECFP_{i}" for i in range(4096)],
        'no-pi': [],
        'no-pol': [],
        'n-ecfp': []
    }
    
    pol_cols = others_select_dict.get(pol)
    torsion_cols = geo_select_dict.get(torsion_selection)
    sd_cols = geo_select_dict.get(sd_selection)
    pi_cols = others_select_dict.get(pi)
    ecfp_cols = others_select_dict.get(ecfp)
    spec_cols = ['DFT Oscillator Strength', 'DFT CD Intensity']
    
    if pol.split('-')[0] != 'no' and pi.split('-')[0] != 'no':
        energy_cols = [
            'Molecule HOMO (eV)', 'Molecule LUMO (eV)', 'Molecule HLG (eV)', 
            'ΔE_HOMO (eV)', 'ΔE_LUMO (eV)', 'Monomer HLG (eV)']
        first_cols.extend(energy_cols)

    col_order = first_cols + pol_cols + torsion_cols + sd_cols + pi_cols + ecfp_cols + spec_cols
    
    df = df[col_order]
    
    df = df.rename(columns = {
        'DFT CD Intensity': 'DFT CD Intensity (a.u.)',
        'DFT Oscillator Strength': 'DFT Oscillator Strength (a.u.)',
    })
    
    new_col_names = []
    for col in df.columns:
        if 'Torsion' in col or 'SD' in col:
            new_col_names.append(col + ' (°)')
        elif 'Polarizability' in col or 'α' in col:
            new_col_names.append(col + ' (Å$^3$)')
        elif 'Dipole' in col:
            new_col_names.append(col + ' (Debye)')
        else:
            new_col_names.append(col)
    
    df.columns = new_col_names
    
    if len(torsion_selection.split('-')[0]) < 2:
        df = df.rename(columns = {
            'Total Torsion Angle_z (°)': 'Total Torsion Angle (°)',
            'Mean Torsion Angle_z (°)': 'Mean Torsion Angle (°)',
            'Inner Mean Torsion Angle_z (°)': 'Inner Mean Torsion Angle (°)',
        })
        
    if len(sd_selection.split('-')[0]) < 2:
        df = df.rename(columns = {
            'SD Torsions_x (°)': 'SD Torsions (°)',
            'SD Difference_x (°)': 'SD Difference (°)'
        })
        
    df = df.dropna().reset_index(drop = True)
    
    print(f'Number of features (m): {len(df.columns)}')
    print(f'Number of data points (n): {len(df)}')
    print(f"Input features: {list(dict.fromkeys([x.split('_')[0] for x in df.drop(['DFT CD Intensity (a.u.)'], axis = 1).columns]))}\n")
    return df

In [ ]:
def load_dataset(df):
    X = df.drop('DFT CD Intensity (a.u.)', axis = 1)
    y = df['DFT CD Intensity (a.u.)']
    
    return X, y

In [ ]:
def scalerize_data(X_train, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.fit_transform(X_test)

    return X_train_scaled, X_test_scaled

In [ ]:
def get_metrics(y_test, y_pred):
    rmse = sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    return rmse, r2

In [ ]:
def get_lasso(df, ft_code, alpha, output_path, output_name, opt_or_not):
    lasso_results = []

    X, y = load_dataset(df)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 4937548)
    X_train_scaled, X_test_scaled = scalerize_data(X_train, X_test)
    
    lasso = Lasso(alpha = alpha)
    lasso.fit(X_train_scaled, y_train)
    y_pred = lasso.predict(X_test_scaled)
    rmse, r2 = get_metrics(y_test, y_pred)

    for i, col in enumerate(X.columns):
        lasso_results.append({'Feature': col, 'Coefficient': lasso.coef_[i]})

    lasso_df = pd.DataFrame(lasso_results, columns = ['Feature', 'Coefficient'])
    lasso_df['Abs. Coefficient'] = lasso_df['Coefficient'].abs()
    lasso_df = lasso_df.sort_values(by = ['Abs. Coefficient'], ascending = False).reset_index(drop = True)

    stats_df = pd.DataFrame({"Alpha": [alpha], "R2": [r2], "RMSE": [rmse]})
    
    if opt_or_not == True:
        lasso_df = lasso_df[lasso_df['Abs. Coefficient'] != 0]
        alpha_name = str(alpha).replace(".","-")
        lasso_df.to_csv(os.path.join(output_path, f"lasso_{alpha_name}_{output_name}.csv"), index = False)
        
        if 'y-ecfp' in ft_code:
            dropecfp_lasso_df = lasso_df[~lasso_df['Feature'].str.contains('ECFP')]
            dropecfp_lasso_df.to_csv(os.path.join(output_path, f"lasso_{alpha_name}_{output_name}_dropecfp.csv"), index = False)
        return lasso_df
    
    else:
        return stats_df

In [ ]:
def get_alpha_df(input_df, ft_code, output_path, output_name):
    all_stats_dfs = []

    for alpha in [0.00001, 0.0001, 0.001, 0.01, 0.1, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 7.5, 10.0]:
        stats_df = get_lasso(input_df, ft_code, alpha, output_path, output_name, False)
        all_stats_dfs.append(stats_df)

    combined_stats_df = pd.concat(all_stats_dfs, ignore_index = True)
    combined_stats_df = combined_stats_df.sort_values(by = "Alpha", ascending = True, ignore_index = True)
    combined_stats_df.to_csv(os.path.join(output_path, f"lasso_stats_{output_name}.csv"), index = False)
    
    return combined_stats_df

In [ ]:
def get_opt_alpha(combined_stats_df):
    r2_max = combined_stats_df['R2'].idxmax()
    rmse_min = combined_stats_df['RMSE'].idxmin()
    if r2_max != rmse_min:
        print(f"Index of R2 max row = {r2_max}. Index of RMSE min row = {rmse_min}")
        print(f"R2 max row: {combined_stats_df.loc[r2_max]}")
        print(f"RMSE min row: {combined_stats_df.loc[rmse_min]}")
        param_choice = input("Use R2 max row alpha or RMSE min row alpha?")
        if param_choice == 'R2':
            opt_alpha = combined_stats_df.loc[r2_max, 'Alpha']
        else:
            opt_alpha = combined_stats_df.loc[rmse_min, 'Alpha']
    else:
        opt_alpha = combined_stats_df.loc[r2_max, 'Alpha']
    
    return opt_alpha

In [ ]:
def fi_graph(features, coefficients, ft_count, output_dir, descriptor, preview):
    figname = f"LASSO_{descriptor}.pdf"
    indices = np.argsort(coefficients)
    num_features = ft_count
    
    selected_indices = indices[-num_features:][::-1]
    selected_coeffs = [coefficients[i] for i in selected_indices][::-1]
    selected_features = [features[i] for i in selected_indices][::-1]
    
    if 'aa' in descriptor:
        for i in range(len(selected_features)):
            if '_x (°)' in selected_features[i]:
                selected_features[i] = selected_features[i].replace('_x (°)', '_0 (°)')
            elif '_y (°)' in selected_features[i]:
                selected_features[i] = selected_features[i].replace('_y (°)', '_1 (°)')
            elif '_z (°)' in selected_features[i]:
                selected_features[i] = selected_features[i].replace('_z (°)', '_2 (°)')
    
    cmap_list = ['#24E4BC', '#00CFF9', '#648FFF', '#785EF0', '#AA42B8', '#DC267F', '#ED2F2C', '#FE6100', '#FF8900', '#FFB000', '#FFE200']
    cmap_list = cmap_list[0:(ft_count-1)]
    custom_cmap = ListedColormap(cmap_list, name = 'cb')
    
            
    my_cmap = plt.get_cmap(name = custom_cmap)
    my_colors = [my_cmap(i / num_features) for i in reversed(range(num_features))]
    
    fig, ax = plt.subplots(figsize = (10, 6), dpi = 300)
    
    bars = ax.barh(range(num_features), selected_coeffs, color = my_colors, edgecolor = 'k', align='center')

    ax.set_yticks(range(num_features))
    ax.set_yticklabels(selected_features, fontsize = 16)
    
    plt.xlabel('Absolute LASSO Coefficient', fontsize = 20)
    plt.xticks(fontsize = 16)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, figname))
    
    if preview == True:
        plt.show()
    else:
        plt.close()

In [ ]:
def main(df, ft_code, output_path, preview):
    output_name = ft_dict.get(ft_code)
    if output_name.endswith('_pt'):
        df = df[df['Oligomer Type'] == 2].reset_index(drop = True)
        
    data_df = df.copy(deep = True)
    data_df = process_df(data_df, ft_code)
    
    combined_stats_df = get_alpha_df(data_df, ft_code, output_path, output_name)
    opt_alpha = get_opt_alpha(combined_stats_df)
    lasso_df = get_lasso(data_df, ft_code, opt_alpha, output_path, output_name, opt_or_not=True)
    
    if 'y-ecfp' in ft_code:
        dropecfp_df = lasso_df[~lasso_df['Feature'].str.contains('ECFP')]
        dropecfp_lasso_features = dropecfp_df['Feature'].tolist()
        dropecfp_lasso_coeffs = dropecfp_df['Abs. Coefficient'].tolist()
        dropecfp_out_name = f"{output_name}_noecfp"
        num_fts = 10 if len(dropecfp_lasso_features) >= 10 else len(dropecfp_lasso_features)
        fi_graph(dropecfp_lasso_features, dropecfp_lasso_coeffs, num_fts, output_path, dropecfp_out_name, preview)
    
    lasso_features = lasso_df['Feature'].tolist()
    lasso_coeffs = lasso_df['Abs. Coefficient'].tolist()
    num_fts = 10 if len(lasso_features) >= 10 else len(lasso_features)
    fi_graph(lasso_features, lasso_coeffs, num_fts, output_path, output_name, preview)

In [ ]:
ft_dict = {
    'old-pol_old-pi_y-ecfp_all-torsion_all-sd': 'ooyaa',
    'old-pol_old-pi_y-ecfp_z-torsion_x-sd': 'ooyzx',
    'old-pol_old-pi_y-ecfp_z-torsion_x-sd_pt': 'ooyzx_pt',
    'new-pol_new-pi_y-ecfp_z-torsion_x-sd': 'nnyzx',
    'new-pol_old-pi_y-ecfp_z-torsion_x-sd': 'noyzx',
}

ft_codes = list(ft_dict.keys())

df = pd.read_csv('clean_dft_df_4096.csv')
output_path = os.path.join(os.getcwd(), 'output', 'test')

In [ ]:
os.makedirs(output_path, exist_ok = True)

In [ ]:
preview = False

for i in range(len(ft_codes)):
    ft_code = ft_codes[i]
    main(df, ft_code, output_path, preview)